# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [10]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [11]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [12]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [13]:
feature_columns = contract["features"]

tree_preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), feature_columns),
], remainder="drop")
scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), feature_columns),
], remainder="drop")

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Escribe aquí tu respuesta.

## 3. Modelos candidatos

In [14]:
models = {
    "decision_tree": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "logistic_regression": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "knn": Pipeline([
        ("preprocessor", scaled_preprocessor),
        ("model", KNeighborsClassifier()),
    ]),
}

## 4. Comparación inicial con validación cruzada

In [15]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
initial_rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring="accuracy", return_train_score=True)
    initial_rows.append({
        "model": name,
        "train_accuracy_mean": scores["train_score"].mean(),
        "validation_accuracy_mean": scores["test_score"].mean(),
        "validation_accuracy_std": scores["test_score"].std(),
    })
initial_comparison = pd.DataFrame(initial_rows).sort_values("validation_accuracy_mean", ascending=False)
display(initial_comparison)

,model,train_accuracy_mean,validation_accuracy_mean,validation_accuracy_std
1,logistic_regression,1.0000,0.979064,0.027713
2,knn,0.9771,0.950739,0.035274
0,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [16]:
param_grids = {
    "decision_tree": {
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "logistic_regression": {"model__C": [0.1, 1.0, 10.0]},
    "knn": {
        "model__n_neighbors": [3, 5, 9],
        "model__weights": ["uniform", "distance"],
    },
}
searches = {}
tuned_rows = []
for name, model in models.items():
    search = GridSearchCV(model, param_grids[name], cv=cv, scoring="accuracy", n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    searches[name] = search
    tuned_rows.append({
        "model": name,
        "best_score": search.best_score_,
        "best_params": search.best_params_,
    })
tuned_comparison = pd.DataFrame(tuned_rows).sort_values("best_score", ascending=False)
display(tuned_comparison)

c:\General\Eafit\Semestre8\IA\si3003-artificial-intelligence\.venv\Lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


,model,best_score,best_params
1,logistic_regression,0.979064,{'model__C': 1.0}
2,knn,0.950739,"{'model__n_neighbors': 5, 'model__weights': 'u..."
0,decision_tree,0.902463,"{'model__max_depth': None, 'model__min_samples..."


## 6. Selección y persistencia del campeón

In [17]:
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")
training_metadata = {
    "champion": champion_name,
    "cv_folds": cv.n_splits,
    "scoring": "accuracy",
    "best_cv_accuracy": float(searches[champion_name].best_score_),
    "best_params": searches[champion_name].best_params_,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str), encoding="utf-8"
)
tuned_comparison.to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Campe?n:", champion_name, training_metadata["best_cv_accuracy"])

Campe?n: logistic_regression 0.9790640394088669


In [18]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

La decisión de seleccionar la Regresión Logística (logistic_regression) como el modelo campeón se fundamenta en los siguientes criterios:   
1. Desempeño (Rendimiento): La regresión logística obtuvo la mejor exactitud (accuracy) media en validación con un 0.979 (97.9%), superando significativamente a KNN (0.950) y al Árbol de Decisión (0.902).   
2. Variabilidad: Presentó la menor desviación estándar durante la validación cruzada inicial (0.0277 frente a 0.0352 de KNN y 0.0586 del árbol). Esto indica que el modelo es el más estable y robusto, garantizando que su rendimiento no depende fuertemente de cómo se dividen los datos de entrenamiento.   
3. Sobreajuste (Overfitting): Aunque la métrica en entrenamiento fue perfecta (1.000), la altísima métrica de validación (~0.979) demuestra que el modelo generaliza excelentemente bien y la brecha entre entrenamiento y validación es mínima. En contraste, el árbol de decisión mostró un claro sobreajuste al tener 1.0 en entrenamiento pero caer drásticamente a 0.902 en validación.   
4. Complejidad: Es un modelo paramétrico muy simple y ligero. Una vez entrenado, las predicciones se calculan mediante una simple combinación lineal, lo que lo hace mucho más eficiente en producción que modelos basados en distancias (como KNN, que requiere guardar todos los datos) o árboles profundos.   
5. Interpretabilidad: La Regresión Logística es un modelo de "caja blanca". Podemos examinar fácilmente los pesos (coeficientes) asignados a cada variable para entender no solo qué características son las más importantes, sino también en qué dirección influyen sobre la predicción. Esto ofrece una gran ventaja de negocio frente a modelos menos interpretables.

## Checklist

- [✓] Todo el procesamiento está dentro de pipelines.
- [✓] Comparé tres familias de modelos.
- [✓] Optimicé hiperparámetros con CV.
- [✓] Elegí el campeón sin usar test.
- [✓] Guardé el pipeline completo y sus metadatos.